In [1]:
# imports and settings

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window, hilbert
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import utils as ut

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

Imports complete.
Settings: height=800, width=1400, font_size=16
Imports complete.


In [18]:
# WAV
folder_path = '../data/croatia/2407_1_600m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list[6:-15]:
    # print(f"Loading file: {file}")
    _data, fs = librosa.load(os.path.join(folder_path, file))
    data.append(_data)
data = np.concatenate(data)
# data = data[80*fs:]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")


# GPX
gpx_file = os.path.join(folder_path, '600m_route.gpx')
gpx_data = gpxpy.parse(open(gpx_file))

# get the first track
track = gpx_data.tracks[0]

# get the first segment
segment = track.segments[0]

rx_location = gpxpy.gpx.GPXWaypoint(44.330008, 14.697141)

distances = []
for point in segment.points:
    distance = rx_location.distance_2d(point)
    distances.append(distance)

distances = np.array(distances)
t_distances = np.linspace(0, len(data)/fs, len(distances))

data final shape: (51597000,) of duration: 2340.0 seconds


In [3]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 8000
norm_size = 5
s2g_quantization_levels = 3
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

welch_detector = ut.WelchDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)

---

In [4]:
_data = data[700*fs:760*fs]

demon, demon_freqs = ut.demon_spectrum(_data, fs, remove_dc=dc, crop_freq=crop_freq)
entropy, entropy_freqs = ut.spectral_entropy(_data, fs, nperseg, overlap, window, remove_dc=dc, crop_freq=crop_freq)
cepstrum_real, cepstrum_real_freqs = ut.cepstrum(_data, fs, nperseg, overlap, window, remove_dc=dc, crop_freq=crop_freq, mode='real')
cepstrum_power, cepstrum_power_freqs = ut.cepstrum(_data, fs, nperseg, overlap, window, remove_dc=dc, crop_freq=crop_freq, mode='power')

features = {
    'demon': {
        'values': demon,
        'freqs': demon_freqs
    },
    'entropy': {
        'values': entropy,
        'freqs': entropy_freqs
    },
    'cepstrum (real)': {
        'values': cepstrum_real,
        'freqs': cepstrum_real_freqs
    },
    'cepstrum (power)': {
        'values': cepstrum_power,
        'freqs': cepstrum_power_freqs
    }
}

ut.plot_spectrogram_with_spectral_features(_data, features, fs, nperseg, overlap, window, dc, crop_freq)


In [12]:
f0 = 631  # Hz

demon, _, _ = ut.demon_spectrum_narrowband(_data, fs, f0, 3, remove_dc=dc, crop_freq=crop_freq, agg='max')
entropy, _, _ = ut.spectral_entropy_narrowband(_data, fs, nperseg, overlap, window,
                                                            target_freq=f0, bandwidth=3,
                                                            remove_dc=dc, crop_freq=crop_freq, agg='max')
cepstrum, _, _ = ut.cepstrum_narrowband(_data, fs, nperseg, overlap, window,
                                                                target_freq=f0, bandwidth=3,
                                                                remove_dc=dc, crop_freq=crop_freq, mode='power', agg='max')

demon, entropy, cepstrum

(np.float32(0.33117366),
 np.float64(7.875466781260874),
 np.float32(9.707439e-06))

---

In [13]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f1_welch_scores = []
f2_welch_scores = []

f1_td_scores = []
f2_td_scores = []

f1_s2g_scores = []
f2_s2g_scores = []

f1_demon_scores = []
f2_demon_scores = []

f1_entropy_scores = []
f2_entropy_scores = []

f1_cepstrum_scores = []
f2_cepstrum_scores = []


f1 = 631  # Hz
f2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    P = welch_detector.get_score_1fb(_data, f1)
    f1_welch_scores.append(P)
    P = welch_detector.get_score_1fb(_data, f2)
    f2_welch_scores.append(P)

    TD = talmon_detector.get_score_1fb(_data, f1)
    f1_td_scores.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f2)
    f2_td_scores.append(TD)

    K = s2g_detector.get_score_1fb(_data, f1)
    f1_s2g_scores.append(K)
    K = s2g_detector.get_score_1fb(_data, f2)
    f2_s2g_scores.append(K)

    demon, _, _ = ut.demon_spectrum_narrowband(_data, fs, f1, 3, remove_dc=dc, crop_freq=crop_freq, agg='max')
    f1_demon_scores.append(demon)
    demon, _, _ = ut.demon_spectrum_narrowband(_data, fs, f2, 3, remove_dc=dc, crop_freq=crop_freq, agg='max')
    f2_demon_scores.append(demon)

    entropy, _, _ = ut.spectral_entropy_narrowband(_data, fs, nperseg, overlap, window,
                                                            target_freq=f1, bandwidth=3,
                                                            remove_dc=dc, crop_freq=crop_freq, agg='max')
    f1_entropy_scores.append(entropy)
    entropy, _, _ = ut.spectral_entropy_narrowband(_data, fs, nperseg, overlap, window,
                                                            target_freq=f2, bandwidth=3,
                                                            remove_dc=dc, crop_freq=crop_freq, agg='max')
    f2_entropy_scores.append(entropy)

    cepstrum_real, _, _ = ut.cepstrum_narrowband(_data, fs, nperseg, overlap, window,
                                                                target_freq=f1, bandwidth=3,
                                                                remove_dc=dc, crop_freq=crop_freq, mode='real', agg='max')
    f1_cepstrum_scores.append(cepstrum_real)
    cepstrum_real, _, _ = ut.cepstrum_narrowband(_data, fs, nperseg, overlap, window,
                                                                target_freq=f2, bandwidth=3,
                                                                remove_dc=dc, crop_freq=crop_freq, mode='real', agg='max')
    f2_cepstrum_scores.append(cepstrum_real)

detectors_taxis = np.linspace(T[0], T[-1], len(f1_s2g_scores))
print("completed all detectors")

completed all detectors


In [ ]:
# fig = make_subplots(rows=8, 
#                     cols=1, 
#                     row_heights=[0.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 
#                     vertical_spacing=0.01, 
#                     shared_xaxes=True,
#                     row_titles=["Spectrogram", "distance","Welch", "Talmon", "S2G", "Demon", "Entropy", "Cepstrum"])

# fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
# fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_welch_scores, name=f'Welch f={f1} Hz', marker_color='blue'), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_welch_scores, name=f'Welch f={f2} Hz', marker_color='green'), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_td_scores, name=f'Talmon f={f1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_td_scores, name=f'Talmon f={f2} Hz', marker_color='green', showlegend=False), row=4, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_s2g_scores, name=f'Score f={f1} Hz', marker_color='blue'), row=5, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_s2g_scores, name=f'Score f={f2} Hz', marker_color='green'), row=5, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_demon_scores, name=f'Demon f={f1} Hz', marker_color='blue'), row=6, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_demon_scores, name=f'Demon f={f2} Hz', marker_color='green'), row=6, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_entropy_scores, name=f'Entropy f={f1} Hz', marker_color='blue'), row=7, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_entropy_scores, name=f'Entropy f={f2} Hz', marker_color='green'), row=7, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_cepstrum_scores, name=f'Cepstrum f={f1} Hz', marker_color='blue'), row=8, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_cepstrum_scores, name=f'Cepstrum f={f2} Hz', marker_color='green'), row=8, col=1)


# fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
# fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
# fig.update_yaxes(title_text="Score", row=3, col=1)
# fig.update_yaxes(title_text="Score", row=4, col=1)
# fig.update_yaxes(title_text="Score", row=5, col=1)
# fig.update_yaxes(title_text="Score", row=6, col=1)
# fig.update_yaxes(title_text="Score", row=7, col=1)
# fig.update_yaxes(title_text="Score", row=8, col=1)
# fig.update_xaxes(title_text="Time (s)", row=8, col=1)

# fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, scores for f0 = {f1}, {f2} Hz")
# fig.show(renderer="browser")

---

In [20]:
# BG Noise WAVs
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

bg_data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(folder_path, file))
    bg_data.append(_data)
bg_data = np.concatenate(bg_data)
print(f"bg_data final shape: {bg_data.shape} of duration: {len(bg_data)/fs} seconds")

# Add BG Noise to data
noise_factor = 10
bg_in_shape = np.concatenate([bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data])
n_data = data + noise_factor * bg_in_shape[:len(data)]
print(f"data final shape: {n_data.shape} of duration: {len(n_data)/fs} seconds")

bg_data final shape: (6615000,) of duration: 300.0 seconds
data final shape: (51597000,) of duration: 2340.0 seconds


In [21]:
nF, nT, nSxx, nphasogram = ut.calc_spectrogram(n_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f1_td_scores_n = []
f2_td_scores_n = []

f1_s2g_scores_n = []
f2_s2g_scores_n = []

f1_demon_scores_n = []
f2_demon_scores_n = []

f1_entropy_scores_n = []
f2_entropy_scores_n = []


f1 = 631  # Hz
f2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = n_data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    TD = talmon_detector.get_score_1fb(_data, f1)
    f1_td_scores_n.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f2)
    f2_td_scores_n.append(TD)

    K = s2g_detector.get_score_1fb(_data, f1)
    f1_s2g_scores_n.append(K)
    K = s2g_detector.get_score_1fb(_data, f2)
    f2_s2g_scores_n.append(K)

    demon, _, _ = ut.demon_spectrum_narrowband(_data, fs, f1, 3, remove_dc=dc, crop_freq=crop_freq, agg='max')
    f1_demon_scores_n.append(demon)
    demon, _, _ = ut.demon_spectrum_narrowband(_data, fs, f2, 3, remove_dc=dc, crop_freq=crop_freq, agg='max')
    f2_demon_scores_n.append(demon)

    entropy, _, _ = ut.spectral_entropy_narrowband(_data, fs, nperseg, overlap, window,
                                                            target_freq=f1, bandwidth=3,
                                                            remove_dc=dc, crop_freq=crop_freq, agg='max')
    f1_entropy_scores_n.append(entropy)
    entropy, _, _ = ut.spectral_entropy_narrowband(_data, fs, nperseg, overlap, window,
                                                            target_freq=f2, bandwidth=3,
                                                            remove_dc=dc, crop_freq=crop_freq, agg='max')
    f2_entropy_scores_n.append(entropy)

detectors_taxis = np.linspace(T[0], T[-1], len(f1_s2g_scores))
print("completed all detectors")

completed all detectors


In [26]:
f1_entropy_scores = -1 * np.array(f1_entropy_scores)
f2_entropy_scores = -1 * np.array(f2_entropy_scores)
f1_entropy_scores_n = -1 * np.array(f1_entropy_scores_n)
f2_entropy_scores_n = -1 * np.array(f2_entropy_scores_n)

In [30]:
fig = make_subplots(rows=6, 
                    cols=2, 
                    row_heights=[0.5, 0.1, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    column_titles=["No Noise", "SNR 1:10"],
                    row_titles=["Spectrogram", "Distance", "Demon", "Correlation", "Entropy", "S2G"])

_ds = max(1, Sxx.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_demon_scores, name=f'Demon f={f1} Hz', marker_color='blue', showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_demon_scores, name=f'Demon f={f2} Hz', marker_color='green', showlegend=False), row=3, col=1)
dix = np.where(detectors_taxis >= 494)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=3, col=1)


fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_td_scores, name=f'Correlation f={f1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_td_scores, name=f'Correlation f={f2} Hz', marker_color='green', showlegend=False), row=4, col=1)
dix = np.where(detectors_taxis >= 154)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=4, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_entropy_scores, name=f'Entropy f={f1} Hz', marker_color='blue', showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_entropy_scores, name=f'Entropy f={f2} Hz', marker_color='green', showlegend=False), row=5, col=1)
dix = np.where(detectors_taxis >= 886)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=5, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_s2g_scores, name=f'Score f={f1} Hz', marker_color='blue'), row=6, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_s2g_scores, name=f'Score f={f2} Hz', marker_color='green'), row=6, col=1)
dix = np.where(detectors_taxis >= 154)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=6, col=1)



fig.add_trace(go.Heatmap(z=nSxx[:, ::_ds], x=nT[::_ds], y=nF, colorscale='viridis', showlegend=False, showscale=False), row=1, col=2)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=2)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_demon_scores_n, name=f'Demon f={f1} Hz', marker_color='blue', showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_demon_scores_n, name=f'Demon f={f2} Hz', marker_color='green', showlegend=False), row=3, col=2)
dix = np.where(detectors_taxis >= 804)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=3, col=2)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_td_scores_n, name=f'Correlation f={f1} Hz', marker_color='blue', showlegend=False), row=4, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_td_scores_n, name=f'Correlation f={f2} Hz', marker_color='green', showlegend=False), row=4, col=2)
dix = np.where(detectors_taxis >= 814)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=4, col=2)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_entropy_scores_n, name=f'Entropy f={f1} Hz', marker_color='blue', showlegend=False), row=5, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_entropy_scores_n, name=f'Entropy f={f2} Hz', marker_color='green', showlegend=False), row=5, col=2)
dix = np.where(detectors_taxis >= 907)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=5, col=2)

fig.add_trace(go.Scatter(x=detectors_taxis, y=f1_s2g_scores_n, name=f'Score f={f1} Hz', marker_color='blue', showlegend=False), row=6, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f2_s2g_scores_n, name=f'Score f={f2} Hz', marker_color='green', showlegend=False), row=6, col=2)
dix = np.where(detectors_taxis >= 494)[0][0]
fig.add_vline(x=detectors_taxis[dix], line_dash="dash", line_color="red", line_width=2, row=6, col=2)



fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)
fig.update_yaxes(title_text="Score", row=6, col=1)
fig.update_xaxes(title_text="Time (s)", row=6, col=1)

fig.update_layout(height=800, width=1400, title_text=f"Scooter experiment 600m, scores for f0 = {f1}, {f2} Hz")
fig.update_annotations(font_size=12)
# fig.show(renderer="browser")
fig.write_image("../results/plots/nb88_600m_w_noise_reference_detectors.pdf")
print("done")

done
